In [1]:
import os
for _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_k, "4")       # Kaggle has 4 vCPUs; the reference runs used 4 BLAS threads as well
import gc, hashlib, json, re, shutil, subprocess, sys, threading, time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import linear_sum_assignment
import lightgbm as lgb

SMOKE = os.environ.get("WEAR_SMOKE", "0") == "1"      # tiny run for testing the code
SMOKE_SUBJECTS = {0, 1, 2, 3, 4}
ABLATION = os.environ.get("WEAR_ABLATION", "1") == "1"   # LOCAL: also score the unchanged public pipeline out-of-fold
# LOCAL: WEAR_CLEAN=1 cuts the sessions without labels off the end of their recordings (seconds kept) and
# moves lagging labels earlier (section 2). Off by default: on the public LB it did not help.
UNLABELLED_TAIL = {"sbj_10": 2566, "sbj_2": 3452} if os.environ.get("WEAR_CLEAN", "0") == "1" else {}
# LOCAL: labels that lag the signals: recording -> (from second, lag in seconds); the labels are moved earlier
LABEL_LAG = {"sbj_7": (1368, 10)} if UNLABELLED_TAIL else {}
SLUG = "3rd-wear-dataset-challenge-hasca-2026"
ON_KAGGLE = Path("/kaggle/input").exists()


def find_input():
    if os.environ.get("WEAR_DATA"):
        return Path(os.environ["WEAR_DATA"])
    for p in (Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG):
        if (p / "test" / "test_meta_data.csv").exists():
            return p
    hits = sorted(Path("/kaggle/input").glob("*/test/test_meta_data.csv")) + \
        sorted(Path("/kaggle/input").glob("*/*/test/test_meta_data.csv"))
    assert hits, "competition data not found: attach the competition or set WEAR_DATA"
    return hits[0].parent.parent


INPUT = find_input()
WORK = Path(os.environ.get("WEAR_WORK", "/kaggle/working" if ON_KAGGLE else "wear_run")).resolve()   # submission.csv
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
if os.environ.get("WEAR_ROOT"):
    ROOT = Path(os.environ["WEAR_ROOT"])
elif ON_KAGGLE and shutil.disk_usage("/tmp").free > 12e9:
    ROOT = Path("/tmp/wear")            # about 6.5 GB of scratch that must not become notebook output
else:
    ROOT = WORK / "_wear"
PROC, RUNS_DIR = ROOT / "proc", ROOT / "runs"
for _p in (ROOT, PROC, RUNS_DIR):
    _p.mkdir(parents=True, exist_ok=True)
CLEANUP = os.environ.get("WEAR_CLEANUP", "1" if ON_KAGGLE else "0") == "1"
WINDOW_BUDGET_MIN = float(os.environ.get("WEAR_WINDOW_BUDGET_MIN", "150"))   # wall-clock cap of the GPU stage

T_START, TIMES = time.time(), {}


def rss_gb():
    try:
        s = Path("/proc/self/status").read_text()
        return tuple(int(re.search(k + r":\s+(\d+)", s).group(1)) / 2 ** 20 for k in ("VmRSS", "VmHWM"))
    except Exception:
        return float("nan"), float("nan")


class stage:
    """logs wall time and memory of a pipeline stage and frees memory afterwards"""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f"\n===== {self.name} | start at {(self.t - T_START) / 60:.1f} min", flush=True)
        return self

    def __exit__(self, *exc):
        gc.collect()
        TIMES[self.name] = time.time() - self.t
        r, h = rss_gb()
        print(f"===== {self.name} | {TIMES[self.name] / 60:.1f} min (total {(time.time() - T_START) / 60:.1f} min) "
              f"| RSS {r:.1f} GB, peak {h:.1f} GB", flush=True)


def error_line(text):
    """the most informative line of a subprocess traceback: the last 'SomeError: message' line"""
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    hits = [l for l in lines if re.match(r"[A-Za-z_][\w.]*(Error|Exception)(:|$)", l)]
    return (hits or lines or ["?"])[-1]


# Is there a GPU that torch can actually use? This runs in a subprocess so that the notebook itself holds no CUDA context.
# A GPU can be visible but unusable: torch.cuda.is_available() is True, yet the torch build has no kernels for it
# (e.g. a P100 = sm_60 with a torch build for sm_75+). The first CUDA op then fails, so the check runs one.
_chk = subprocess.run([sys.executable, "-c",
                       "import json, torch; assert torch.cuda.is_available(), 'torch.cuda.is_available() is False'; "
                       "torch.ones(8, device='cuda').sum().item(); "
                       "print(json.dumps({'n': torch.cuda.device_count(), 'name': torch.cuda.get_device_name(0), "
                       "'cap': torch.cuda.get_device_capability(0), 'torch': torch.__version__}))"],
                      capture_output=True, text=True)
GPU_OK = _chk.returncode == 0
GPU_INFO = json.loads(_chk.stdout.strip().splitlines()[-1]) if GPU_OK else {"n": 0}
N_GPU = GPU_INFO["n"]
print(f"input {INPUT}\nwork {WORK}\nscratch {ROOT}\nSMOKE={SMOKE} GPU={GPU_INFO if GPU_OK else 'none'}")
if not GPU_OK:
    print("WARNING: no usable GPU (" + error_line(_chk.stderr) + ").\n"
          "The window models fall back to a very reduced CPU run (with CUDA hidden from the training subprocesses), "
          "so the score will be far below 0.903. Select the GPU T4 x2 accelerator to reproduce the LB score.")

input /kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026
work /kaggle/working
scratch /tmp/wear
SMOKE=False GPU={'n': 2, 'name': 'Tesla T4', 'cap': [7, 5], 'torch': '2.11.0+cu128'}


In [2]:
N_CLS, WIN, VLEN = 19, 50, 15
CFG = {"imu_w": 0.3, "lp": (10, 0.1, 0.5, 0.9), "link_alpha": 0.5, "link_iters": 10, "sharpen_T": 0.5, "per_ex": 97,
       "null_min": 0.05}
TRAIN_SETS = {0: 2, 14: 2}   # train subjects 0 and 14 recorded the exercise circuit twice; every test subject once


def macro_f1(y, pred):
    cm = np.bincount(y * N_CLS + pred, minlength=N_CLS * N_CLS).reshape(N_CLS, N_CLS)
    tp = np.diag(cm)
    denom = cm.sum(0) + cm.sum(1)
    present = denom > 0
    return float((2 * tp[present] / denom[present]).mean())


def mean_logp(logps):
    return np.log(np.mean([np.exp(x) for x in logps], 0) + 1e-8)


def blend(fus, imu, wi):
    x = fus + wi * imu if imu is not None else fus
    x = x - x.max(1, keepdims=True)
    return x - np.log(np.exp(x).sum(1, keepdims=True))


def label_prop(P0, emb, grp, k=10, tau=0.1, use_p=0.5, alpha=0.9, iters=20):
    """per subject: kNN graph on (video similarity + prediction similarity), then probability propagation"""
    E = emb.astype(np.float32)
    E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-6
    F = np.empty_like(P0)
    for g in np.unique(grp):
        ii = np.flatnonzero(grp == g)
        n = len(ii)
        Q = np.sqrt(P0[ii]).astype(np.float32)
        S = E[ii] @ E[ii].T + use_p * (Q @ Q.T)
        np.fill_diagonal(S, -np.inf)
        nb = np.argpartition(-S, k, axis=1)[:, :k]
        rows = np.repeat(np.arange(n), k)
        vals = S[rows, nb.reshape(-1)]
        W = sp.csr_matrix((np.exp((vals - vals.max()) / tau), (rows, nb.reshape(-1))), shape=(n, n))
        W = W.maximum(W.T)
        d_ = np.asarray(W.sum(1)).ravel()
        d_[d_ == 0] = 1
        Dm = sp.diags(1 / np.sqrt(d_))
        Wn = Dm @ W @ Dm
        f0 = P0[ii]
        f = f0.copy()
        for _ in range(iters):
            f = alpha * (Wn @ f) + (1 - alpha) * f0
        F[ii] = f
    return F


def link_smooth(P, succ, score, alpha, iters):
    """mix each window with its linked successor/predecessor (weights sigmoid(link log-odds))"""
    m = succ >= 0
    src, dst = np.flatnonzero(m), succ[m]
    wt = 1 / (1 + np.exp(-np.clip(score[m], -20, 20)))
    W = sp.csr_matrix((np.r_[wt, wt], (np.r_[src, dst], np.r_[dst, src])), shape=(len(P), len(P)))
    d_ = np.asarray(W.sum(1)).ravel()
    cur = P.copy()
    for _ in range(iters):
        nb = (W @ cur) / np.maximum(d_[:, None], 1e-8)
        mix = np.where(d_[:, None] > 0, (1 - alpha) * P + alpha * nb, P)
        cur = mix / mix.sum(1, keepdims=True)
    return cur


def calibrate(P, sbj, sets, per_ex, null_min, iters=50):
    """Sinkhorn per subject: each exercise column sums to per_ex x (number of sets), null takes the rest"""
    out = np.empty_like(P)
    for s in np.unique(sbj):
        ii = np.flatnonzero(sbj == s)
        n = len(ii)
        null = max(n - 18 * per_ex * sets.get(int(s), 1), null_min * n)
        t_ = np.full(P.shape[1], (n - null) / 18)
        t_[0] = null
        Q = P[ii].copy()
        for _ in range(iters):
            Q *= (t_ / np.maximum(Q.sum(0), 1e-9))[None]
            Q /= Q.sum(1, keepdims=True)
        out[ii] = Q
    return out


def context_baseline(d):
    P = label_prop(np.exp(d["logp"]), d["emb"], d["grp"], *CFG["lp"])
    P = P / P.sum(1, keepdims=True)
    return link_smooth(P, d["succ"], d["score"], CFG["link_alpha"], CFG["link_iters"])


def finish(P, d):
    """pre-calibration probabilities (N,19) -> sharpen -> per-subject count calibration -> labels"""
    Q = np.clip(P, 1e-12, None) ** (1 / CFG["sharpen_T"])
    Q = calibrate(Q / Q.sum(1, keepdims=True), d["sbj"], d["sets"], CFG["per_ex"], CFG["null_min"])
    return Q.argmax(1)

In [3]:
CLASSES = ["null", "jogging", "jogging (rotating arms)", "jogging (skipping)", "jogging (sidesteps)",
           "jogging (butt-kicks)", "stretching (triceps)", "stretching (lunging)", "stretching (shoulders)",
           "stretching (hamstrings)", "stretching (lumbar rotation)", "push-ups", "push-ups (complex)",
           "sit-ups", "sit-ups (complex)", "burpees", "lunges", "lunges (complex)", "bench-dips"]
LABEL_IDX = {c: i for i, c in enumerate(CLASSES)}
SENSORS = ["right_arm", "right_leg", "left_leg", "left_arm"]
ACC_COLS = [f"{s}_acc_{a}" for s in SENSORS for a in "xyz"]


def prep():
    recs = []
    csvs = sorted((INPUT / "train" / "inertial_feat").glob("*.csv"))     # this order defines the recording index
    if SMOKE:
        csvs = [c for c in csvs if int(c.stem.split("_")[1]) in SMOKE_SUBJECTS]
    for csv in csvs:
        stem = csv.stem
        df = pd.read_csv(csv, usecols=["sbj_id", *ACC_COLS, "label"], dtype={"label": str})
        acc = df[ACC_COLS].to_numpy(np.float32)
        valid = ~np.isnan(acc).reshape(-1, 4, 3).any(2)
        n_nan = int(np.isnan(acc).sum())
        if n_nan:
            acc = pd.DataFrame(acc).interpolate(limit_direction="both").fillna(0).to_numpy(np.float32)
        labels = df["label"].fillna("null").astype(str).str.strip()
        unknown = set(labels.unique()) - set(CLASSES)
        assert not unknown, f"{stem}: unknown labels {unknown}"
        lab = labels.map(LABEL_IDX).to_numpy(np.int8)
        vid = np.load(INPUT / "train" / "videomae_feat" / f"{stem}.npy").astype(np.float16)
        if stem in LABEL_LAG:            # LOCAL: in this session the labels come 10 s after the activity (section 1)
            t0, lag = LABEL_LAG[stem][0] * 50, LABEL_LAG[stem][1] * 50
            lab[t0:len(lab) - lag] = lab[t0 + lag:].copy()
            lab[len(lab) - lag:] = 0
        if stem in UNLABELLED_TAIL:      # LOCAL: the last session of this recording carries no labels (section 1)
            sec = UNLABELLED_TAIL[stem]
            acc, lab, valid, vid = acc[:sec * 50], lab[:sec * 50], valid[:sec * 50], vid[:sec * 30]
        np.save(PROC / f"{stem}_acc.npy", acc)
        np.save(PROC / f"{stem}_lab.npy", lab)
        np.save(PROC / f"{stem}_valid.npy", valid)
        np.save(PROC / f"{stem}_vid.npy", vid)
        recs.append({"stem": stem, "sbj": int(df["sbj_id"].iloc[0]), "T": len(acc), "F": len(vid)})
        print(f"{stem}: T={len(acc)} ({len(acc) / 50:.0f}s) F={len(vid)} nan={n_nan}", flush=True)
        del df, acc, vid
    (PROC / "recordings.json").write_text(json.dumps(recs, indent=1))

    meta = pd.read_csv(INPUT / "test" / "test_meta_data.csv")
    t_acc = np.load(INPUT / "test" / "test_inertial_data.npy").astype(np.float32)
    t_vid = np.load(INPUT / "test" / "test_videomae_data.npy")
    print("test shapes:", t_acc.shape, t_vid.shape, t_vid.dtype)
    if t_vid.shape[1] == 768:
        t_vid = t_vid.transpose(0, 2, 1)
    loc_col = "sensor_location" if "sensor_location" in meta else "inertial_sensor_location"
    np.save(PROC / "test_acc.npy", t_acc)
    np.save(PROC / "test_vid.npy", np.ascontiguousarray(t_vid).astype(np.float16))
    np.save(PROC / "test_sensor.npy", meta[loc_col].map({s: i for i, s in enumerate(SENSORS)}).to_numpy(np.int8))
    np.save(PROC / "test_id.npy", meta["id"].to_numpy())
    np.save(PROC / "test_sbj.npy", meta["sbj_id"].to_numpy(np.int16))
    print(meta[loc_col].value_counts().to_dict(), meta["sbj_id"].value_counts().sort_index().to_dict())
    return recs


with stage("W1 preprocessing"):
    if not (ROOT / ".done_prep").exists():
        recs = prep()
        (ROOT / ".done_prep").touch()
    recs = json.loads((PROC / "recordings.json").read_text())
    if not SMOKE:
        _cut = (3564 - UNLABELLED_TAIL.get("sbj_2", 3564)) + (3985 - UNLABELLED_TAIL.get("sbj_10", 3985))   # LOCAL: seconds cut off
        assert len(recs) == 24 and sum(r["T"] for r in recs) == 3466400 - 50 * _cut and sum(r["F"] for r in recs) == 2079840 - 30 * _cut
        assert all(r["F"] * 5 == r["T"] * 3 for r in recs)
    print(f"{len(recs)} recordings, {len({r['sbj'] for r in recs})} subjects")


===== W1 preprocessing | start at 0.2 min
sbj_0: T=139725 (2794s) F=83835 nan=0
sbj_0_2: T=112050 (2241s) F=67230 nan=0
sbj_1: T=138900 (2778s) F=83340 nan=0
sbj_10: T=199250 (3985s) F=119550 nan=154176
sbj_11: T=111100 (2222s) F=66660 nan=0
sbj_12: T=163500 (3270s) F=98100 nan=0
sbj_13: T=198900 (3978s) F=119340 nan=0
sbj_14: T=160050 (3201s) F=96030 nan=0
sbj_14_2: T=167750 (3355s) F=100650 nan=0
sbj_15: T=130150 (2603s) F=78090 nan=0
sbj_16: T=175500 (3510s) F=105300 nan=0
sbj_17: T=154550 (3091s) F=92730 nan=0
sbj_18: T=131600 (2632s) F=78960 nan=0
sbj_19: T=105450 (2109s) F=63270 nan=0
sbj_2: T=178200 (3564s) F=106920 nan=0
sbj_20: T=95400 (1908s) F=57240 nan=0
sbj_21: T=127800 (2556s) F=76680 nan=0
sbj_3: T=208550 (4171s) F=125130 nan=0
sbj_4: T=159575 (3192s) F=95745 nan=0
sbj_5: T=107525 (2150s) F=64515 nan=0
sbj_6: T=125175 (2504s) F=75105 nan=0
sbj_7: T=142650 (2853s) F=85590 nan=0
sbj_8: T=124750 (2495s) F=74850 nan=0
sbj_9: T=108300 (2166s) F=64980 nan=0
test shapes: (1223

In [4]:
STRIDE, FOLDS = 5, 5


def oof_windows():
    recs = json.loads((PROC / "recordings.json").read_text())
    W_rec, W_start, W_y, W_sbj, W_ok = [], [], [], [], []
    for i, r in enumerate(recs):
        lab = np.load(PROC / f"{r['stem']}_lab.npy").astype(np.int64)
        val = np.load(PROC / f"{r['stem']}_valid.npy")
        T, Fr = r["T"], r["F"]
        starts = np.arange(0, T - WIN + 1, STRIDE)
        starts = starts[starts * 3 // 5 + 8 + VLEN <= Fr]
        cs = np.zeros((T + 1, N_CLS), np.int32)
        np.cumsum(np.eye(N_CLS, dtype=np.int32)[lab], axis=0, out=cs[1:])
        cnt = cs[starts + WIN] - cs[starts]
        miss = np.zeros((T + 1, 4), np.int32)
        np.cumsum(~val, axis=0, out=miss[1:])
        W_ok.append((miss[starts + WIN] - miss[starts]) == 0)
        W_rec.append(np.full(len(starts), i))
        W_start.append(starts)
        W_y.append(cnt.argmax(1))
        W_sbj.append(np.full(len(starts), r["sbj"]))
    W_rec, W_start, W_y, W_sbj, W_ok = map(np.concatenate, (W_rec, W_start, W_y, W_sbj, W_ok))
    keep = W_ok.any(1)
    W_rec, W_start, W_y, W_sbj, W_ok = (x[keep] for x in (W_rec, W_start, W_y, W_sbj, W_ok))

    sbjs, sbj_n = np.unique(W_sbj, return_counts=True)
    fold_of, load = {}, np.zeros(FOLDS)
    for s in sbjs[np.argsort(-sbj_n)]:
        f = int(load.argmin())
        fold_of[s] = f
        load[f] += sbj_n[sbjs == s][0]
    W_fold = np.array([fold_of[s] for s in W_sbj])

    out = {k: [] for k in ("rec", "start", "sensor", "y", "sbj", "fold")}
    for fold in range(FOLDS):
        idx = np.flatnonzero(W_fold == fold)
        idx = idx[W_start[idx] % WIN == 0]
        rng = np.random.default_rng(1234 + fold)
        cp = np.cumsum(W_ok[idx] / W_ok[idx].sum(1, keepdims=True), 1)
        sensor = (rng.random(len(idx))[:, None] > cp).sum(1)
        for k, v in (("rec", W_rec[idx]), ("start", W_start[idx]), ("sensor", sensor), ("y", W_y[idx]),
                     ("sbj", W_sbj[idx]), ("fold", np.full(len(idx), fold))):
            out[k].append(v)
    out = {k: np.concatenate(v) for k, v in out.items()}
    return recs, out


def load_inputs(recs, w):
    """per-window IMU of the chosen limb (N,50,3) float32 and video (N,15,768) float16"""
    n = len(w["rec"])
    acc = np.zeros((n, WIN, 3), np.float32)
    vid = np.zeros((n, VLEN, 768), np.float16)
    for i, r in enumerate(recs):
        ii = np.flatnonzero(w["rec"] == i)
        if not len(ii):
            continue
        a = np.load(PROC / f"{r['stem']}_acc.npy").reshape(-1, 4, 3)
        v = np.load(PROC / f"{r['stem']}_vid.npy", mmap_mode="r")
        st = w["start"][ii]
        acc[ii] = a[st[:, None] + np.arange(WIN), w["sensor"][ii][:, None]]
        vid[ii] = v[(st * 3 // 5)[:, None] + 8 + np.arange(VLEN)]
    return acc, vid


with stage("W2 test-like OOF windows"):
    if not (ROOT / ".done_simwin").exists():
        _recs, _w = oof_windows()
        _acc, _vid = load_inputs(_recs, _w)
        np.save(ROOT / "sim_acc.npy", _acc)
        np.save(ROOT / "sim_vid.npy", _vid)
        np.savez(ROOT / "sim_meta.npz", **_w)
        del _acc, _vid, _w
        (ROOT / ".done_simwin").touch()
    sim = {k: v.astype(np.int64) for k, v in dict(np.load(ROOT / "sim_meta.npz")).items()}
    fold_of = {int(s): int(f) for s, f in zip(sim["sbj"], sim["fold"])}
    print("rows", len(sim["y"]), "folds", {f: sorted(s for s in fold_of if fold_of[s] == f) for f in range(FOLDS)})
    if not SMOKE:
        assert len(sim["y"]) == (69326 if not UNLABELLED_TAIL else 67795), len(sim["y"])   # LOCAL
        assert UNLABELLED_TAIL or np.bincount(sim["fold"]).tolist() == [14353, 12995, 14625, 12768, 14585]
        assert UNLABELLED_TAIL or hashlib.sha1(sim["sensor"].tobytes()).hexdigest()[:12] == "f81d90085315"
        assert UNLABELLED_TAIL or hashlib.sha1(np.stack([sim["rec"], sim["start"]]).tobytes()).hexdigest()[:12] == "e4177f2589d7"


===== W2 test-like OOF windows | start at 3.8 min
rows 69326 folds {0: [5, 14, 17, 21], 1: [0, 4, 9, 15], 2: [3, 7, 11, 12, 19], 3: [1, 8, 10, 16], 4: [2, 6, 13, 18, 20]}
===== W2 test-like OOF windows | 0.1 min (total 3.9 min) | RSS 0.5 GB, peak 3.0 GB


In [5]:
%%writefile train_window.py
"""Window classifier, 5-fold by subject. This is a verbatim copy of our train.py with Kaggle patches marked 'PATCH':
 (a) mixed precision: bf16 on Ampere+ (as in the original runs), fp16 autocast + GradScaler on T4 (sm_75), and fp32
     on older GPUs, on CPU and when WEAR_AMP=0
 (b) memory-mapped per-recording video: only the concatenated copy is held in RAM
 (c) the class-balanced sampler uses a precomputed CDF, which gives the same indices as rng.choice(n, bs, p=pw)
Outputs (--out): oof_raw.npz (OOF log-probs of the test-like 1-s tiles + video embedding), test_logp_raw.npy (mean of
the 5 fold models' log-softmax), test_emb.npy. oof.npz / test_logp.npy / bias.npy / model_f*.pt are not used later.
"""
import argparse
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

p = argparse.ArgumentParser()
p.add_argument("--data", default="/workspace/proc")
p.add_argument("--out", default="/workspace/runs/v1")
p.add_argument("--folds", type=int, default=5)
p.add_argument("--epochs", type=int, default=24)
p.add_argument("--steps", type=int, default=300)
p.add_argument("--bs", type=int, default=512)
p.add_argument("--lr", type=float, default=1e-3)
p.add_argument("--dim", type=int, default=256)
p.add_argument("--seed", type=int, default=0)
p.add_argument("--ema", type=float, default=0.999)
p.add_argument("--vnorm", default="global", choices=["global", "subject"])  # video feature normalisation unit
p.add_argument("--anorm", default="global", choices=["global", "subject"])  # IMU normalisation unit
p.add_argument("--run_folds", default="")  # e.g. "0,2" (empty = all)
p.add_argument("--eval_every", type=int, default=4)
p.add_argument("--vid_arch", default="pool", choices=["pool", "tf"])  # pool: mean/std pooling + MLP
p.add_argument("--vdrop", type=float, default=0.5)  # video input feature dropout
p.add_argument("--knn", default="1,5,10,20,40")  # neighbour-smoothing k candidates (logging only; 1 = off)
p.add_argument("--only", default="none", choices=["none", "imu", "video"])  # single-modality model
args = p.parse_args()

N_CLS, WIN, VLEN, STRIDE = 19, 50, 15, 5
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# PATCH (a): bf16 autocast is unsupported or emulated on T4 / P100 -> capability switch
CAP = torch.cuda.get_device_capability() if dev.type == "cuda" else (0, 0)
USE_AMP = CAP >= (7, 0)                                             # T4 (7,5): fp16 tensor cores; P100 (6,0): fp32
USE_AMP = USE_AMP and os.environ.get("WEAR_AMP", "1") != "0"          # the driver retries a failed run in fp32
AMP_DTYPE = torch.float16 if (7, 0) <= CAP < (8, 0) else torch.bfloat16
torch.ones(8, device=dev).sum().item()                              # fails fast if torch has no kernels for this GPU
if dev.type == "cuda":
    torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
out_dir = Path(args.out)
out_dir.mkdir(parents=True, exist_ok=True)
log_f = open(out_dir / "log.txt", "a")


def make_scaler(enabled):  # PATCH (a)
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    log_f.write(s + "\n")
    log_f.flush()


log(f"device {dev} capability {CAP} precision {AMP_DTYPE if USE_AMP else torch.float32} torch {torch.__version__}")


def macro_f1(y, pred):
    cm = np.bincount(y * N_CLS + pred, minlength=N_CLS * N_CLS).reshape(N_CLS, N_CLS)
    tp = np.diag(cm)
    denom = cm.sum(0) + cm.sum(1)
    present = denom > 0
    return float((2 * tp[present] / denom[present]).mean())


# ---------------- data ----------------
D = Path(args.data)
recs = json.loads((D / "recordings.json").read_text())
acc_l, lab_l, vid_l, val_l = [], [], [], []
for r in recs:
    acc_l.append(np.load(D / f"{r['stem']}_acc.npy"))
    val_l.append(np.load(D / f"{r['stem']}_valid.npy"))
    lab_l.append(np.load(D / f"{r['stem']}_lab.npy").astype(np.int64))
    vid_l.append(np.load(D / f"{r['stem']}_vid.npy", mmap_mode="r"))  # PATCH (b)
acc_off = np.cumsum([0] + [len(a) for a in acc_l])[:-1]
vid_off = np.cumsum([0] + [len(v) for v in vid_l])[:-1]
acc_all = np.concatenate(acc_l)
vid_all = np.concatenate(vid_l)

test_acc_np = np.load(D / "test_acc.npy")
test_vid_np = np.load(D / "test_vid.npy")
test_sensor_np = np.load(D / "test_sensor.npy").astype(np.int64)
test_sbj_np = np.load(D / "test_sbj.npy").astype(np.int64)

# normalisation groups: train subjects + test subjects (global mode: all groups share one set of stats)
SB = sorted({r["sbj"] for r in recs})
TS = sorted(set(test_sbj_np.tolist()))
G = len(SB) + len(TS)
test_grp_np = np.array([len(SB) + TS.index(t) for t in test_sbj_np])


def acc_stats(x, ok):
    """x (n,4,3), ok (n,4) -> per-sensor mean/std (4,3)"""
    m, sd = np.zeros((4, 3)), np.ones((4, 3))
    for k in range(4):
        xs = x[ok[:, k], k].astype(np.float64)
        if len(xs) > 100:
            m[k], sd[k] = xs.mean(0), xs.std(0) + 1e-3
    return m, sd


def vid_stats(v):
    v = v.astype(np.float32)
    return v.mean(0), v.std(0) + 1e-3


AM, AS = np.zeros((G, 4, 3)), np.ones((G, 4, 3))
VM, VS = np.zeros((G, 768)), np.ones((G, 768))
if args.anorm == "global":
    AM[:], AS[:] = acc_stats(acc_all.reshape(-1, 4, 3)[::5], np.concatenate(val_l)[::5])
else:
    for gi, sb in enumerate(SB):
        ii = [i for i, r in enumerate(recs) if r["sbj"] == sb]
        AM[gi], AS[gi] = acc_stats(np.concatenate([acc_l[i] for i in ii]).reshape(-1, 4, 3), np.concatenate([val_l[i] for i in ii]))
    for ti, t in enumerate(TS):
        m = test_sbj_np == t
        x = np.zeros((m.sum(), WIN, 4, 3), np.float32)
        ok = np.zeros((m.sum(), WIN, 4), bool)
        x[np.arange(m.sum()), :, test_sensor_np[m]] = test_acc_np[m]
        ok[np.arange(m.sum()), :, test_sensor_np[m]] = True
        AM[len(SB) + ti], AS[len(SB) + ti] = acc_stats(x.reshape(-1, 4, 3), ok.reshape(-1, 4))
if args.vnorm == "global":
    VM[:], VS[:] = vid_stats(vid_all[::7])
else:
    for gi, sb in enumerate(SB):
        VM[gi], VS[gi] = vid_stats(np.concatenate([vid_l[i][::3] for i, r in enumerate(recs) if r["sbj"] == sb]))
    for ti, t in enumerate(TS):
        VM[len(SB) + ti], VS[len(SB) + ti] = vid_stats(test_vid_np[test_sbj_np == t].reshape(-1, 768))
AM, AS, VM, VS = (torch.tensor(x, dtype=torch.float32, device=dev) for x in (AM, AS, VM, VS))
ACC = torch.from_numpy(acc_all).to(dev)
VID = torch.from_numpy(vid_all).to(dev)
del acc_all, vid_all

# window list: starts every 0.1 s, label = majority label inside the window
W_rec, W_start, W_y, W_pur, W_sbj, W_ok = [], [], [], [], [], []
for i, r in enumerate(recs):
    T, Fr = len(acc_l[i]), len(vid_l[i])
    starts = np.arange(0, T - WIN + 1, STRIDE)
    starts = starts[starts * 3 // 5 + 8 + VLEN <= Fr]
    cs = np.zeros((T + 1, N_CLS), np.int32)
    np.cumsum(np.eye(N_CLS, dtype=np.int32)[lab_l[i]], axis=0, out=cs[1:])
    cnt = cs[starts + WIN] - cs[starts]
    miss = np.zeros((T + 1, 4), np.int32)
    np.cumsum(~val_l[i], axis=0, out=miss[1:])
    W_ok.append((miss[starts + WIN] - miss[starts]) == 0)  # (n,4) sensors without missing data in the window
    W_rec.append(np.full(len(starts), i))
    W_start.append(starts)
    W_y.append(cnt.argmax(1))
    W_pur.append(cnt.max(1) / WIN)
    W_sbj.append(np.full(len(starts), r["sbj"]))
W_rec, W_start, W_y, W_pur, W_sbj, W_ok = map(np.concatenate, (W_rec, W_start, W_y, W_pur, W_sbj, W_ok))
keep = W_ok.any(1)
W_rec, W_start, W_y, W_pur, W_sbj, W_ok = (x[keep] for x in (W_rec, W_start, W_y, W_pur, W_sbj, W_ok))
W_ok_t = torch.as_tensor(W_ok, device=dev)
W_grp = np.array([SB.index(s) for s in W_sbj])
log(f"(window, sensor) pairs with missing data excluded: {int((~W_ok).sum())}")
log(f"windows={len(W_y)} recordings={len(recs)} subjects={len(np.unique(W_sbj))}")
log("class counts:", np.bincount(W_y, minlength=N_CLS).tolist())

acc_off_t = torch.tensor(acc_off, device=dev)
vid_off_t = torch.tensor(vid_off, device=dev)
ar50 = torch.arange(WIN, device=dev)
ar15 = torch.arange(VLEN, device=dev)


def rand_rot(n, max_deg=20.0):
    axis = F.normalize(torch.randn(n, 3, device=dev), dim=1)
    ang = (torch.rand(n, device=dev) * 2 - 1) * math.radians(max_deg)
    K = torch.zeros(n, 3, 3, device=dev)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 2] = -axis[:, 2], axis[:, 1], -axis[:, 0]
    K = K - K.transpose(1, 2)
    s, c = ang.sin()[:, None, None], ang.cos()[:, None, None]
    return torch.eye(3, device=dev) + s * K + (1 - c) * (K @ K)


def prep_inputs(a_raw, v_raw, sensor, g, train):
    """a_raw (B,50,3) in g, v_raw (B,15,768) f16, sensor (B,), g (B,) normalisation group -> model inputs"""
    B = a_raw.shape[0]
    a_raw = a_raw.float()
    if train:
        a_raw = a_raw @ rand_rot(B)
        a_raw = a_raw * (0.9 + 0.2 * torch.rand(B, 1, 1, device=dev))
        a_raw = a_raw + 0.02 * torch.randn_like(a_raw)
    mag = a_raw.norm(dim=-1, keepdim=True) - 1.0
    a = (a_raw - AM[g, sensor][:, None]) / AS[g, sensor][:, None]
    a = torch.cat([a, mag], -1)
    v = (v_raw.float() - VM[g][:, None]) / VS[g][:, None]
    has_a = torch.ones(B, device=dev)
    has_v = torch.ones(B, device=dev)
    if args.only != "none":
        if args.only == "imu":
            v = torch.zeros_like(v)
        else:
            a = torch.zeros_like(a)
        if train:
            v = v + 0.1 * torch.randn_like(v) * (args.only == "video")
        return a, v, has_a * (args.only == "imu"), has_v * (args.only == "video")
    if train:
        v = v + 0.1 * torch.randn_like(v)
        v = v * (torch.rand(B, VLEN, 1, device=dev) > 0.1)
        r = torch.rand(B, device=dev)
        has_v = (r >= 0.2).float()          # 20%: no video
        has_a = ((r < 0.2) | (r >= 0.3)).float()  # 10%: no IMU
        v = v * has_v[:, None, None]
        a = a * has_a[:, None, None]
    return a, v, has_a, has_v


def gather_train(idx, sensor):
    rec = torch.as_tensor(W_rec[idx], device=dev)
    st = torch.as_tensor(W_start[idx], device=dev)
    B = len(idx)
    a = ACC[acc_off_t[rec][:, None] + st[:, None] + ar50].view(B, WIN, 4, 3)
    a = a[torch.arange(B, device=dev), :, sensor]
    v = VID[vid_off_t[rec][:, None] + (st * 3 // 5)[:, None] + 8 + ar15]
    return a, v


# ---------------- model ----------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, k, d=1):
        super().__init__()
        self.c = nn.Conv1d(cin, cout, k, padding=d * (k // 2), dilation=d)
        self.bn = nn.BatchNorm1d(cout)
        self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        return F.gelu(self.bn(self.c(x))) + self.skip(x)


class Net(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.sens = nn.Embedding(4, 32)
        self.imu = nn.Sequential(ConvBlock(4 + 32, 64, 7), ConvBlock(64, 128, 5), ConvBlock(128, 128, 5, 2),
                                 ConvBlock(128, d, 3, 4), ConvBlock(d, d, 3, 8))
        self.imu_out = nn.Sequential(nn.Linear(2 * d, d), nn.GELU())
        self.vin = nn.Sequential(nn.Dropout(args.vdrop), nn.Linear(768, d), nn.GELU(), nn.Dropout(0.2))
        if args.vid_arch == "tf":
            self.vpos = nn.Parameter(torch.zeros(1, VLEN, d))
            layer = nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, activation="gelu", batch_first=True, norm_first=True)
            self.vtf = nn.TransformerEncoder(layer, 2)
        self.vid_out = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.3))
        self.fuse = nn.Sequential(nn.Linear(2 * d + 32, d), nn.GELU(), nn.Dropout(0.3), nn.Linear(d, N_CLS))
        self.h_imu = nn.Linear(d, N_CLS)
        self.h_vid = nn.Linear(d, N_CLS)

    def forward(self, a, v, sensor):
        e = self.sens(sensor)
        x = torch.cat([a.transpose(1, 2), e[:, :, None].expand(-1, -1, a.shape[1])], 1)
        h = self.imu(x)
        hi = self.imu_out(torch.cat([h.mean(-1), h.amax(-1)], 1))
        if args.vid_arch == "tf":
            z = self.vtf(self.vin(v) + self.vpos)
            hv = self.vid_out(torch.cat([z.mean(1), z.amax(1)], 1))
        else:
            z = self.vin(v)
            hv = self.vid_out(torch.cat([z.mean(1), z.std(1)], 1))
        return self.fuse(torch.cat([hi, hv, e], 1)), self.h_imu(hi), self.h_vid(hv)


@torch.no_grad()
def predict(model, a_raw, v_raw, sensor, g, bs=2048, heads=False):
    model.eval()
    outs = []
    for i in range(0, len(sensor), bs):
        a, v, _, _ = prep_inputs(a_raw[i:i + bs], v_raw[i:i + bs], sensor[i:i + bs], g[i:i + bs], False)
        with torch.autocast(dev.type, dtype=AMP_DTYPE, enabled=USE_AMP):  # PATCH (a)
            logits = model(a, v, sensor[i:i + bs])
        outs.append(torch.stack([F.log_softmax(x.float(), 1) for x in logits]))
    out = torch.cat(outs, 1).cpu().numpy()  # (3, N, C): fusion, imu-only, video-only
    return out if heads else out[0]


def val_set(idx, rng):
    """test-like: every non-overlapping 1-s window with one random limb that has no missing data"""
    idx = idx[W_start[idx] % WIN == 0]
    cp = np.cumsum(W_ok[idx] / W_ok[idx].sum(1, keepdims=True), 1)
    sensor = torch.as_tensor((rng.random(len(idx))[:, None] > cp).sum(1), device=dev)
    a, v = gather_train(idx, sensor)
    return a, v, sensor, torch.as_tensor(W_grp[idx], device=dev), W_y[idx]


@torch.no_grad()
def vid_emb(v_raw, g, bs=4096):
    """for neighbour search: window mean of the normalised video features"""
    return torch.cat([((v_raw[i:i + bs].float() - VM[g[i:i + bs]][:, None]) / VS[g[i:i + bs]][:, None]).mean(1)
                      for i in range(0, len(g), bs)])


@torch.no_grad()
def knn_smooth(logp, emb, grp, k):
    """average the probabilities of the k most similar windows (by video, same subject, including itself)"""
    if k <= 1:
        return logp
    out = np.empty_like(logp)
    P = torch.from_numpy(np.exp(logp)).to(dev)
    E = F.normalize(emb.float(), dim=1)
    for gv in np.unique(grp):
        ii = np.flatnonzero(grp == gv)
        ti = torch.as_tensor(ii, device=dev)
        nb = (E[ti] @ E[ti].T).topk(min(k, len(ii)), dim=1).indices
        out[ii] = torch.log(P[ti][nb].mean(1) + 1e-8).cpu().numpy()
    return out


# ---------------- folds (by subject, balanced window counts) ----------------
sbjs, sbj_n = np.unique(W_sbj, return_counts=True)
fold_of, fold_load = {}, np.zeros(args.folds)
for s in sbjs[np.argsort(-sbj_n)]:
    f = int(fold_load.argmin())
    fold_of[s] = f
    fold_load[f] += sbj_n[sbjs == s][0]
W_fold = np.array([fold_of[s] for s in W_sbj])
log("folds:", {f: sorted(int(s) for s in sbjs if fold_of[s] == f) for f in range(args.folds)})

test_acc = torch.from_numpy(test_acc_np).to(dev)
test_vid = torch.from_numpy(test_vid_np).to(dev)
test_sensor = torch.from_numpy(test_sensor_np).to(dev)
test_grp = torch.from_numpy(test_grp_np).to(dev)
run_folds = [int(f) for f in args.run_folds.split(",")] if args.run_folds else list(range(args.folds))

freq = np.bincount(W_y, minlength=N_CLS).astype(np.float64)
cls_w = (1.0 / np.sqrt(freq))[W_y]  # sample with class frequency^-0.5

oof_logp, oof_y, oof_emb, oof_grp, test_logps = [], [], [], [], []
t0 = time.time()
for fold in run_folds:
    torch.manual_seed(args.seed * 100 + fold)
    rng = np.random.default_rng(args.seed * 100 + fold)
    tr_idx = np.flatnonzero(W_fold != fold)
    va_idx = np.flatnonzero(W_fold == fold)
    pw = cls_w[tr_idx] / cls_w[tr_idx].sum()
    cdf = pw.cumsum()  # PATCH (c): what rng.choice(len(tr_idx), bs, p=pw) computes on every call
    cdf /= cdf[-1]
    va = val_set(va_idx, np.random.default_rng(1234 + fold))

    model = Net(args.dim).to(dev)
    ema = torch.optim.swa_utils.AveragedModel(model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(args.ema),
                                              use_buffers=True)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.05)
    scaler = make_scaler(USE_AMP and AMP_DTYPE == torch.float16)  # PATCH (a)
    total = args.epochs * args.steps
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=total, pct_start=0.1)
    step = 0
    for ep in range(args.epochs):
        model.train()
        tl = 0.0
        for _ in range(args.steps):
            idx = tr_idx[cdf.searchsorted(rng.random(args.bs), side="right")]  # PATCH (c)
            sensor = torch.multinomial(W_ok_t[torch.as_tensor(idx, device=dev)].float(), 1).squeeze(1)
            a_raw, v_raw = gather_train(idx, sensor)
            g = torch.as_tensor(W_grp[idx], device=dev)
            a, v, has_a, has_v = prep_inputs(a_raw, v_raw, sensor, g, True)
            y = torch.as_tensor(W_y[idx], device=dev)
            with torch.autocast(dev.type, dtype=AMP_DTYPE, enabled=USE_AMP):  # PATCH (a)
                out, oi, ov = model(a, v, sensor)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            li = F.cross_entropy(oi.float(), y, label_smoothing=0.1, reduction="none")
            lv = F.cross_entropy(ov.float(), y, label_smoothing=0.1, reduction="none")
            loss = loss + 0.3 * (li * has_a).sum() / has_a.sum().clamp(min=1) + 0.3 * (lv * has_v).sum() / has_v.sum().clamp(min=1)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()  # PATCH (a): pass-through when the scaler is disabled
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update_parameters(model)
            tl += loss.item()
            step += 1
        if (ep + 1) % args.eval_every == 0 or ep == args.epochs - 1:
            lp3 = predict(ema.module, *va[:4], heads=True)
            f1s = [macro_f1(va[4], x.argmax(1)) for x in lp3]
            log(f"fold {fold} ep {ep+1}/{args.epochs} loss {tl/args.steps:.4f} val_macroF1(ema) fusion {f1s[0]:.4f} "
                f"imu {f1s[1]:.4f} video {f1s[2]:.4f} [{time.time()-t0:.0f}s]")
    lp = predict(ema.module, *va[:4])
    oof_logp.append(lp)
    oof_y.append(va[4])
    oof_emb.append(vid_emb(va[1], va[3]))
    oof_grp.append(va[3].cpu().numpy())
    test_logps.append(predict(ema.module, test_acc, test_vid, test_sensor, test_grp))
    torch.save(ema.module.state_dict(), out_dir / f"model_f{fold}.pt")
    log(f"fold {fold} final val_macroF1 {macro_f1(va[4], lp.argmax(1)):.4f}")

oof_logp = np.concatenate(oof_logp)
oof_y = np.concatenate(oof_y)
oof_emb = torch.cat(oof_emb)
oof_grp = np.concatenate(oof_grp)
log(f"OOF macroF1 (no bias) {macro_f1(oof_y, oof_logp.argmax(1)):.4f}")
np.savez(out_dir / "oof_raw.npz", logp=oof_logp, y=oof_y, grp=oof_grp, emb=oof_emb.half().cpu().numpy())

# neighbour smoothing: choose k on OOF (logging only; the pipeline uses the raw log-probs)
best_k, best_f = 1, -1.0
for k in [int(x) for x in args.knn.split(",")]:
    f = macro_f1(oof_y, knn_smooth(oof_logp, oof_emb, oof_grp, k).argmax(1))
    log(f"knn k={k}: OOF macroF1 {f:.4f}")
    if f > best_f:
        best_k, best_f = k, f
log(f"chosen k={best_k}")
oof_logp = knn_smooth(oof_logp, oof_emb, oof_grp, best_k)

# per-class logit bias tuned on OOF macro F1 (logging only)
bias = np.zeros(N_CLS)
grid = np.linspace(-1.5, 1.5, 31)
best = macro_f1(oof_y, oof_logp.argmax(1))
for it in range(3):
    for c in range(N_CLS):
        for g in grid:
            b2 = bias.copy()
            b2[c] = g
            f = macro_f1(oof_y, (oof_logp + b2).argmax(1))
            if f > best:
                best, bias = f, b2
    log(f"bias tuning iter {it}: OOF macroF1 {best:.4f}")
log("bias:", np.round(bias, 2).tolist())

test_logp_raw = np.mean(test_logps, 0)
np.save(out_dir / "test_logp_raw.npy", test_logp_raw)
test_emb = vid_emb(test_vid, test_grp)
np.save(out_dir / "test_emb.npy", test_emb.half().cpu().numpy())
test_logp = knn_smooth(test_logp_raw, test_emb, test_grp_np, best_k)
np.savez(out_dir / "oof.npz", logp=oof_logp, y=oof_y)
np.save(out_dir / "test_logp.npy", test_logp)
np.save(out_dir / "bias.npy", bias)
log("test pred dist (with bias):", np.bincount((test_logp + bias).argmax(1), minlength=N_CLS).tolist())
log(f"done in {time.time()-t0:.0f}s")

Writing train_window.py


In [6]:
TRAIN_COMMON = ["--data", str(PROC), "--vnorm", "subject", "--anorm", "subject", "--vid_arch", "pool"]
FULL_RUNS = [   # FORK: two extra fusion seeds, one per GPU
    ("final_s0", "--seed 20 --epochs 12 --eval_every 4 --knn 1,3,5"),
    ("final_s1", "--seed 21 --epochs 12 --eval_every 4 --knn 1,3,5"),
]
SEED_OFF = int(os.environ.get("WEAR_SEED", "0"))      # LOCAL: other seeds for an independent run
if SEED_OFF:
    FULL_RUNS = [(n, re.sub(r"--seed (\d+)", lambda m: f"--seed {int(m.group(1)) + 10 * SEED_OFF}", a)) for n, a in FULL_RUNS]
CPU_RUNS = [  # emergency fallback without a usable GPU: heavily under-trained, only keeps the notebook running
    ("final_s0", "--seed 0 --epochs 4 --steps 100 --bs 256 --eval_every 4 --knn 1"),
    ("imu_s0", "--seed 0 --epochs 4 --steps 100 --bs 256 --eval_every 4 --only imu --knn 1"),
]
SMOKE_RUNS = [
    ("final_s0", "--seed 0 --epochs 1 --steps 3 --bs 64 --eval_every 1 --knn 1"),
    ("imu_s0", "--seed 0 --epochs 1 --steps 3 --bs 64 --eval_every 1 --only imu --knn 1"),
]
ESSENTIAL = ("final_s0",)   # FORK


def run_complete(name):
    return all((RUNS_DIR / name / f).exists() for f in ("oof_raw.npz", "test_logp_raw.npy", "test_emb.npy"))


def run_steps(a):
    arg = lambda k, dflt: int(re.search(rf"--{k} (\d+)", a).group(1)) if f"--{k} " in a else dflt
    return 5 * arg("steps", 300) * arg("epochs", 24)


def run_window(name, a, gpu, amp=True):
    env = dict(os.environ)
    if gpu is not None:
        env.update(CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS="1")
    else:
        env["CUDA_VISIBLE_DEVICES"] = ""        # CPU fallback: hide a GPU that is visible but unusable (see the GPU check)
    env["WEAR_AMP"] = "1" if amp else "0"       # 0 = fp32 training (mixed precision off)
    cmd = [sys.executable, str(WORK / "train_window.py"), *TRAIN_COMMON, *a.split(), "--out", str(RUNS_DIR / name)]
    ts = time.time()
    with open(RUNS_DIR / f"{name}.out", "w") as fo:
        rc = subprocess.run(cmd, stdout=fo, stderr=subprocess.STDOUT, env=env).returncode
    text = (RUNS_DIR / f"{name}.out").read_text()
    tail = text.strip().splitlines()
    oof = [l for l in tail if l.startswith("OOF macroF1")]
    print(f"[{name}] gpu={gpu} rc={rc} {(time.time() - ts) / 60:.1f} min | {oof[0] if oof else ''}", flush=True)
    if rc != 0:
        print("\n".join(tail[-25:]) + f"\n[{name}] error: {error_line(text)}", flush=True)
    return rc


def run_queue(queue, gpu, t0, failed):
    sec_per_step, amp = None, gpu is not None
    for name, a in queue:
        steps = run_steps(a)
        if sec_per_step and name not in ESSENTIAL and \
                (time.time() - t0 + steps * sec_per_step + 90) / 60 > WINDOW_BUDGET_MIN:
            print(f"time guard: skip {name} (would need about {steps * sec_per_step / 60:.0f} min)", flush=True)
            continue
        ts = time.time()
        rc = run_window(name, a, gpu, amp)
        if rc != 0 and amp:
            # safety net: a failure on the mixed-precision path (fp16 + GradScaler on T4) must not sink the notebook,
            # so retry once in fp32 and keep fp32 for the rest of this queue
            print(f"[{name}] retrying in fp32; mixed precision is off for the rest of this queue", flush=True)
            amp, ts = False, time.time()
            rc = run_window(name, a, gpu, amp)
        if rc != 0:
            failed.append(name)
            continue
        sec_per_step = max(time.time() - ts - 60, 1.0) / steps      # about 60 s per run is loading/predicting


with stage("W3 window models"):
    RUNS = SMOKE_RUNS if SMOKE else (FULL_RUNS if GPU_OK else CPU_RUNS)
    todo = [(n, a) for n, a in RUNS if not run_complete(n)]
    print("runs to train:", [n for n, _ in todo], "| already done:", [n for n, _ in RUNS if run_complete(n)])
    failed, t0 = [], time.time()
    if GPU_OK and N_GPU >= 2 and len(todo) > 1:
        queues, load = [[], []], [0, 0]                    # longest-first split into two balanced queues
        for n, a in sorted(todo, key=lambda x: -run_steps(x[1])):
            q = int(np.argmin(load))
            queues[q].append((n, a))
            load[q] += run_steps(a)
        prio = [n for n, _ in RUNS]
        queues = [sorted(q, key=lambda x: prio.index(x[0])) for q in queues]
        print("queues:", [[n for n, _ in q] for q in queues], "steps", load)
        threads = [threading.Thread(target=run_queue, args=(q, g, t0, failed)) for g, q in enumerate(queues)]
        for th in threads:
            th.start()
            time.sleep(30)                                 # stagger the 3.2 GB data loads
        for th in threads:
            th.join()
    else:
        run_queue(todo, 0 if GPU_OK else None, t0, failed)
    assert not [n for n in failed if n in ESSENTIAL], f"essential window run failed: {failed}"
    FUSION = [RUNS_DIR / n for n in ("final_s0", "final_s1", "final_s2", "final_s3") if run_complete(n)]
    IMU = [RUNS_DIR / n for n in ("imu_s0", "imu_s1") if run_complete(n)]
    assert len(FUSION) == 2, FUSION   # FORK
    print("fusion runs:", [p.name for p in FUSION], "| IMU-only runs:", [p.name for p in IMU])


===== W3 window models | start at 3.9 min
runs to train: ['final_s0', 'final_s1'] | already done: []
queues: [['final_s0'], ['final_s1']] steps [18000, 18000]
[final_s0] gpu=0 rc=0 7.7 min | OOF macroF1 (no bias) 0.7119
[final_s1] gpu=1 rc=0 7.5 min | OOF macroF1 (no bias) 0.7075
fusion runs: ['final_s0', 'final_s1'] | IMU-only runs: []
===== W3 window models | 8.0 min (total 11.9 min) | RSS 0.5 GB, peak 3.0 GB


In [7]:

# ---- FORK: s7's window blend rebuilt with the two new fusion seeds
import glob
_old = sorted(glob.glob("/kaggle/input/**/keep4/runs/*.npz", recursive=True))
print("s7 runs found:", [os.path.basename(p) for p in _old])
_fo = [p for p in _old if os.path.basename(p).startswith("final_s")]
_io = [p for p in _old if os.path.basename(p).startswith("imu_s")]
assert len(_fo) == 8 and len(_io) == 4, (_fo, _io)
_rn = lambda x: x - np.log(np.exp(x - x.max(1, keepdims=True)).sum(1, keepdims=True)) - x.max(1, keepdims=True)
_s7 = sorted(glob.glob("/kaggle/input/**/keep4/stage.npz", recursive=True))[0]
_st = np.load(_s7, allow_pickle=True)
_new_o = [np.load(r / "oof_raw.npz")["logp"].astype(np.float32) for r in FUSION]
_new_t = [_rn(np.load(r / "test_logp_raw.npy").astype(np.float32)) for r in FUSION]
assert (np.load(FUSION[0] / "oof_raw.npz")["y"] == _st["oof_y"]).all()
_fo_o = [np.load(p)["oof_logp"].astype(np.float32) for p in _fo]; _fo_t = [_rn(np.load(p)["test_logp"].astype(np.float32)) for p in _fo]
_io_o = [np.load(p)["oof_logp"].astype(np.float32) for p in _io]; _io_t = [_rn(np.load(p)["test_logp"].astype(np.float32)) for p in _io]
_chk = blend(mean_logp(_fo_o), mean_logp(_io_o), CFG["imu_w"])
print(f"s7 window blend rebuilt from its runs: max abs diff {np.abs(_chk - _st['oof_logp']).max():.4f}")
_lo = blend(mean_logp(_fo_o + _new_o), mean_logp(_io_o), CFG["imu_w"])
_lt = blend(mean_logp(_fo_t + _new_t), mean_logp(_io_t), CFG["imu_w"])
print(f"window OOF macro F1: s7 {macro_f1(_st['oof_y'], _st['oof_logp'].argmax(1)):.4f} -> with new seeds {macro_f1(_st['oof_y'], _lo.argmax(1)):.4f}")
_out = WORK / "wseed"; _out.mkdir(exist_ok=True)
np.savez(_out / "window_logp.npz", oof_logp=_lo.astype(np.float32), test_logp=_lt.astype(np.float32),
         new_oof=np.stack(_new_o).astype(np.float16), new_test=np.stack(_new_t).astype(np.float16))
print("saved", _out / "window_logp.npz")


s7 runs found: ['final_s0.npz', 'final_s1.npz', 'final_s2.npz', 'final_s3.npz', 'final_s4.npz', 'final_s5.npz', 'final_s6.npz', 'final_s7.npz', 'imu_s0.npz', 'imu_s1.npz', 'imu_s2.npz', 'imu_s3.npz']
s7 window blend rebuilt from its runs: max abs diff 0.0022
window OOF macro F1: s7 0.7264 -> with new seeds 0.7267
saved /kaggle/working/wseed/window_logp.npz
